[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Bigdata-com/bigdata-getting-started/blob/main/notebooks/04_research.ipynb)

# 04 · Research Agent — AI-powered analysis

The **[Bigdata.com](https://bigdata.com)** Research Agent (`POST /v1/research-agent`)
goes beyond retrieval: it **plans** a research approach, runs multiple searches,
and **synthesises a cited answer** grounded in the underlying documents.

**What it's good for**
- One-shot answers to analytical questions ("What are the key risks for X?")
- Multi-step research that would otherwise need many manual searches
- Grounded, source-referenced narratives you can drop into a report

**Important:** this endpoint **streams** its progress as Server-Sent Events
(SSE) and lives on the **`agents.bigdata.com`** host. Each `data:` line is a JSON
event; the message `type` tells you what it is (`THINKING`, `PLANNING`,
`ACTION`, `ANSWER`, `COMPLETE`, ...). We collect the `ANSWER` chunks for the
final response.

In [1]:
import os
import requests

# Your API key is read from the BIGDATA_API_KEY environment variable.
# Never hard-code keys in a notebook you plan to share or commit.
API_KEY = os.environ["BIGDATA_API_KEY"]

# Bigdata exposes two hosts:
#   - api.bigdata.com    -> Knowledge Graph & Search Service (deterministic APIs)
#   - agents.bigdata.com -> Research Agent & Workflows (AI agents, streamed)
API_BASE = "https://api.bigdata.com"
AGENTS_BASE = "https://agents.bigdata.com"

# The key must be sent in the X-API-KEY header on every request.
HEADERS = {"X-API-KEY": API_KEY, "Content-Type": "application/json"}

### A helper to consume the SSE stream
It yields parsed events and concatenates the `ANSWER` text into the final answer.

In [2]:
import json

def run_research(message, research_effort="lite", tools_configs=None, show_progress=True):
    """Stream the Research Agent and return the final answer text.
    research_effort: 'lite' (quick) or 'standard' (thorough)."""
    payload = {"message": message, "research_effort": research_effort}
    if tools_configs:
        payload["tools_configs"] = tools_configs

    answer_parts, plan_printed = [], set()
    with requests.post(f"{AGENTS_BASE}/v1/research-agent", headers=HEADERS, json=payload,
                       stream=True, timeout=600) as r:
        r.raise_for_status()
        for line in r.iter_lines(decode_unicode=True):
            if not line or not line.startswith("data:"):
                continue
            event = json.loads(line[len("data:"):].strip())
            msg = event.get("message", {})
            mtype = msg.get("type")
            if mtype == "PLANNING" and show_progress:
                for step in msg.get("plan", {}).get("steps", []):
                    if step["description"] not in plan_printed:
                        plan_printed.add(step["description"])
                        print("  plan:", step["description"])
            elif mtype == "ACTION" and show_progress:
                print("  action:", msg.get("tool_name"))
            elif mtype == "ANSWER":
                answer_parts.append(msg.get("content", ""))
    return "".join(answer_parts)

### Example 1 — A quick (lite) research question

In [3]:
answer = run_research("What are the main risks Apple highlighted recently?", research_effort="lite")
print("\n=== ANSWER ===\n")
print(answer)

  action: search_companies


  action: search_filings



=== ANSWER ===

In its Q4 2025 earnings report, Apple highlighted several material risks that could negatively affect its business, financial condition, operating results, and stock price . Among the primary factors identified were:

*   **Reliance on a Single Product Category:** Apple generates a significant portion of its net sales from a single product category. A decline in demand for these products would have a significant impact on the company's net sales and gross margins .
*   **Foreign Exchange Fluctuations:** The company's financial performance is exposed to risks associated with the value of the U.S. dollar relative to other currencies. This affects non-U.S. dollar-denominated sales, cost of sales, and operating expenses worldwide. Fluctuations in these rates have in the past, and could in the future, materially and adversely affect the company's gross margins .


### Example 2 — Constrain the research with filters
`tools_configs.search.query_filters` lets you scope the agent's searches — here
to a specific entity and time period. Use `ranking_parameters` to bias freshness
or source quality (1-10).

In [4]:
apple = requests.post(f"{API_BASE}/v1/knowledge-graph/companies", headers=HEADERS,
                      json={"query": "Apple", "types": ["PUBLIC"]}, timeout=30).json()["results"][0]["id"]

answer = run_research(
    "Summarise the most recent demand trends discussed.",
    research_effort="lite",
    tools_configs={
        "search": {
            "query_filters": {
                "entities": {"any_of": [apple]},
                "period": {"start": "2025-01-01T00:00:00Z", "end": "2025-12-31T23:59:59Z"},
            },
            "ranking_parameters": {"freshness_boost": 5},
        }
    },
)
print("\n=== ANSWER ===\n")
print(answer)

  action: get_earnings_calendar


  action: search



=== ANSWER ===

Recent demand trends for Apple, based on data leading into mid-2026, reflect a period of strong performance for core hardware, tempered by macroeconomic headwinds and increased competitive pressure.

### Strong iPhone and Mac Performance
The primary driver of demand in late 2025 and early 2026 was the **iPhone 17 series**. The product launch saw robust consumer interest in both the US and China, which helped Apple set revenue records and put the company on track to retake the title of the world’s largest smartphone maker . Key drivers for this demand included an inflection point in the smartphone replacement cycle and a focus on performance, camera upgrades, and design improvements rather than AI-specific features .

Similarly, the **Mac lineup** demonstrated strength, with 13% year-over-year revenue growth reported in September 2025, largely supported by the popularity of the MacBook Air .

### Market Headwinds and Challenges
Despite the successful launch of recent pr

## Notes
- **`research_effort`**: `lite` for fast answers, `standard` for deeper analysis.
- **`model_name`**: `base` (default) or `pro` for harder reasoning tasks.
- **Follow-ups**: set `persistence_mode="enabled"` and reuse the returned
  `chat_id` to continue a conversation.
- **Structured output**: pass `structured_output_schema` (a JSON Schema) to get
  machine-readable extraction instead of prose.

**Next:** [`05_workflows.ipynb`](05_workflows.ipynb)

_Source: [Bigdata.com](https://bigdata.com) · Research docs: https://docs.bigdata.com/api-reference/research-agent/research-agent_